[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# Models and Fields


## What you will be able to do

Write the model classes a peewee program starts with: a base class that names the database once, and
models that inherit it. Read the `CREATE TABLE` peewee writes from the fields, and know which of
`null`, `unique` and `index` is a rule the database enforces rather than a habit of your code. Make
two columns unique together. Open and close a connection on purpose, and say what `autoconnect`
does. Recognize the failures of the first hour: a model bound to no database, a table that was never
made, a table made twice, a required column left empty, and a database nobody opened.


## The idea

### The problem

The **Why Peewee** notebook wrote two model classes without explaining them, which is the right
order to meet
them in and no way to leave them. A model has to say three things, and each of them has a way of
going wrong that costs a beginner an hour.

It has to say **which database**. peewee does not have one global connection that models find by
themselves: a model carries a database, or it carries nothing, and a query built from a model that
carries nothing has nowhere to go. That is the most common first failure, and the message says
exactly what happened once you know how to read it.

It has to say **what the columns are**, which is the fields, and the fields say more than a type.
`null=True` is a column that accepts nothing; `unique=True` is a constraint the database enforces on
every writer; `index=True` is a second statement entirely. Confusing any of those with a check your
own code does is how a program ends up with rules that hold only while it is the only thing writing.

And the table has to **exist**. Declaring a class creates nothing: `create_tables` does, once, and
running it again without `safe=True` is an error rather than a no-op.

### What a model is

> A **model** is a class that inherits `peewee.Model`. Its class attributes are **fields**, which
> become columns, and an inner class **`Meta`** carries what belongs to the table rather than to one
> column: **`database`**, **`table_name`**, **`indexes`** and **`primary_key`**. Every peewee program
> defines one **base model** that names the database in its `Meta` and lets every other model inherit
> it. **`db.create_tables([...])`** writes the `CREATE TABLE` for each, in an order that satisfies
> the foreign keys, and does nothing for a table that is already there unless told otherwise. A
> database object is **lazy**: it opens a connection when a query needs one, unless `autoconnect` is
> off.

### Why it works that way

- **A model carries its database.** That is what makes the same models usable against two databases
  later, by rebinding them, which the **SQLite and PostgreSQL** notebook does.
- **The fields are the schema.** peewee writes the `CREATE TABLE` from them, so a field's arguments
  are the table's rules, and a rule you want the database to keep has to be one of them.
- **`null=True` is not a default.** It says the column accepts nothing at all. A column without it
  refuses a missing value, whatever your code intended.
- **An index is its own statement.** `index=True` adds a `CREATE INDEX` beside the table, which is
  why it never appears inside the `CREATE TABLE`.
- **The connection is opened when it is needed.** Which is convenient in a script and worth taking
  control of in a program that runs for a long time, with `connect`, `close` and a context manager.

### Where this shows up

The first file of every peewee project, which is a `models.py` with a base class at the top. The
**Object-Oriented Python** guide is where a class inside a class and inheritance are taught. The
**Migrations** notebook is where a change to one of these classes becomes a change to a database
that already has rows in it.

### What this notebook covers

- The base class every peewee program starts with
- What each field becomes as a column
- `null`, `unique` and `index`: which of them the database keeps
- Two columns that have to be unique together
- What a value is when it comes back
- The connection, opened and closed on purpose
- The catalog's models, finished
- Five failures of the first hour

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from peewee import CharField, IntegerField, Model, SqliteDatabase

db = SqliteDatabase(":memory:")


class Base(Model):
    class Meta:
        database = db


class Author(Base):                             # the database comes from the base class
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField(index=True)


print("bound to the database:", Author._meta.database is db)
print("the columns:", [field.column_name for field in Author._meta.sorted_fields])
for index in Author._schema._create_indexes():
    print(index.query()[0])
```

```
bound to the database: True
the columns: ['id', 'name', 'first_book']
CREATE UNIQUE INDEX IF NOT EXISTS "author_name" ON "author" ("name")
CREATE INDEX IF NOT EXISTS "author_first_book" ON "author" ("first_book")
```

Two classes, and only one of them mentions the database. The `id` is there without being declared,
and neither `unique=True` nor `index=True` is inside the table: both became statements beside it,
and the only difference between them is the word `UNIQUE`, which is the distinction this notebook is
mostly about.


## Setup

Eight imports, peewee installed and pinned, the catalog, and three helpers.

- `peewee` is the library, and `Model`, the field classes, `CompositeKey` and `SqliteDatabase`, from
  it, are what a model is written with
- `subprocess`, `sys`, `version` and `PackageNotFoundError` install peewee 4.5.1 where the version is
  not that, as on Colab, whose 4.4.0 words some of these messages differently
- `datetime` and `Decimal` are the Python types two of the fields take and give back
- `re` takes a memory address out of a message, which the first of the Common errors carries
- `AUTHORS` and `BOOKS` are the catalog, loaded at the end of the worked examples
- `sql` prints the SQL a query will send, `message` prints an error's class and text, and `table_sql`
  prints the `CREATE TABLE` a model describes with any index that goes with it

`db` is one `SqliteDatabase(":memory:")`, made here and used by every model below. A database in
memory lasts as long as the connection does, which is what makes it right for a notebook: nothing is
left behind, and a rerun starts from nothing. The **Migrations** and **A Small Catalog** notebooks
need a file instead, and say so.


In [1]:
import datetime
import re
import subprocess
import sys
from decimal import Decimal
from importlib.metadata import PackageNotFoundError, version

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import (BooleanField, CharField, CompositeKey, DateField, DateTimeField, DecimalField,
                    ForeignKeyField, IntegerField, Model, SqliteDatabase, TextField)

AUTHORS = [                                                         # name, the year of the first book
    ("Ursula Vance", 2014),
    ("Marco Pietra", 2009),
    ("Ines O'Brien", 1998),
    ("Kofi Mensah", 2015),
]

BOOKS = [                                                           # title, author, year, pages
    ("The Salt Road", "Ursula Vance", 2014, 312),
    ("Nightjar", "Ursula Vance", 2018, 244),
    ("The Quiet Engine", "Ursula Vance", 2021, 398),
    ("Stone and Tide", "Marco Pietra", 2009, 501),
    ("The Lantern Keeper", "Marco Pietra", 2016, 276),
    ("Riverwork", "Marco Pietra", 2022, 189),
    ("A Careful Fire", "Ines O'Brien", 1998, 420),
    ("The Long Field", "Ines O'Brien", 2004, 355),
    ("Winter Harbour", "Ines O'Brien", 2011, 263),
    ("The Drum Line", "Kofi Mensah", 2015, 198),
    ("Harmattan", "Kofi Mensah", 2019, 331),
    ("Small Machines", "Kofi Mensah", 2023, 287),
]

def sql(query):
    """The SQL a query will send, and the values that go with it, on one line."""
    statement, values = query.sql()
    return " ".join(statement.split()) + (f"  {values}" if values else "")

def message(error):
    """An error's class and text, without the memory address that makes no two runs agree."""
    return f"{type(error).__module__}.{type(error).__name__}: {re.sub(r'0x[0-9a-f]+', '0x...', str(error))}"

def table_sql(model):
    """The CREATE TABLE peewee writes for a model, and any index that goes with it."""
    written = [model._schema._create_table().query()[0]]
    return "\n".join(written + [index.query()[0] for index in model._schema._create_indexes()])

db = SqliteDatabase(":memory:")

print("peewee", peewee.__version__, "| the database:", db.database, "| open:", not db.is_closed())


peewee 4.5.1 | the database: :memory: | open: False


## Worked examples

### The base class every peewee program starts with

One class names the database, and every model inherits it:


In [2]:
class CatalogModel(Model):
    """Every model in the catalog names the database once, here."""

    class Meta:
        database = db


class Author(CatalogModel):
    name = CharField(max_length=60, unique=True)
    first_book = IntegerField()


print("the base's database :", CatalogModel._meta.database is db)
print("the author's        :", Author._meta.database is db)
print("the table's name    :", Author._meta.table_name)


the base's database : True
the author's        : True
the table's name    : author


`Meta` is inherited, so `Author` is bound without saying so, and a project with thirty models names
the database once. The table's name came from the class name in lower case, and `Meta.table_name`
is how to say otherwise, which matters against a database somebody else designed.

Without that inheritance a model has no database at all, which is the first of the Common errors and
the most common first hour in peewee.

### What each field becomes as a column

A model with one field of most kinds, and the table peewee writes from it:


In [3]:
class Loan(CatalogModel):
    borrower = CharField(max_length=60)
    note = TextField(null=True)
    copies = IntegerField()
    fee = DecimalField(max_digits=6, decimal_places=2)
    taken_on = DateField(index=True)
    due_at = DateTimeField()
    returned = BooleanField(default=False)


print(table_sql(Loan))


CREATE TABLE IF NOT EXISTS "loan" ("id" INTEGER NOT NULL PRIMARY KEY, "borrower" VARCHAR(60) NOT NULL, "note" TEXT, "copies" INTEGER NOT NULL, "fee" DECIMAL(6, 2) NOT NULL, "taken_on" DATE NOT NULL, "due_at" DATETIME NOT NULL, "returned" INTEGER NOT NULL)
CREATE INDEX IF NOT EXISTS "loan_taken_on" ON "loan" ("taken_on")


Seven fields and eight columns, because `id` is added for you: peewee gives a model an `AutoField`
primary key unless one is declared. `CharField(max_length=60)` is a `VARCHAR(60)` and `TextField` is
`TEXT` with no length; `DecimalField` carries its digits; `DateField` and `DateTimeField` are stored
by SQLite as text, which the round trip below is about; and `BooleanField` has a `default` that is
Python's, since there is no `DEFAULT` in that `CREATE TABLE` at all. The one index came from
`index=True` on the date, as a statement of its own.

| What you write | The column | Enforced by |
|---|---|---|
| `CharField(max_length=60)` | `VARCHAR(60)` | nothing on SQLite, the length on other databases |
| `TextField(null=True)` | `TEXT` with no `NOT NULL` | the database |
| `DecimalField(max_digits=6, decimal_places=2)` | `DECIMAL(6, 2)` | the database |
| `BooleanField(default=False)` | `INTEGER NOT NULL` | Python fills the default in |
| `index=True` | a `CREATE INDEX` of its own | the database |
| `unique=True` | a `CREATE UNIQUE INDEX` of its own | the database |

`EnumField` and `IntEnumField` are in `playhouse.fields` rather than in peewee itself, which is worth
knowing before reaching for an enumerated column and finding nothing.

### null, unique and index: which of them the database keeps

The three are easy to read as the same kind of thing, and only one of them is about Python:


In [4]:
db.create_tables([Author, Loan])

Author.create(name="Ursula Vance", first_book=2014)
print("a second author of the same name:")
try:
    Author.create(name="Ursula Vance", first_book=2020)
except peewee.IntegrityError as error:
    print("   ", message(error))

print("a loan with no note :", Loan.create(borrower="a", copies=1, fee=Decimal("0.00"),
                                           taken_on=datetime.date(2026, 1, 5),
                                           due_at=datetime.datetime(2026, 2, 5, 9, 0)).note)
print("the default applied :", Loan.get_by_id(1).returned)


a second author of the same name:
    peewee.IntegrityError: UNIQUE constraint failed: author.name
a loan with no note : None
the default applied : False


`unique=True` is enforced by the database, on every writer, for as long as the table exists, through
the unique index it made beside the table rather than through anything inside it. `null=True` let the
note be missing. `default=False` was filled in by peewee when the object was
built, and is nowhere in the table, which the **Creating and Changing Rows** notebook shows mattering
when a row is written by something other than a model.

The one that is not about Python at all is `index=True`: it changes no rule, only what the database
does to answer a query, and the **Selecting Rows** notebook is where that is measured.

### Two columns that have to be unique together

A rule about two columns has nowhere to live on either one, so it goes in `Meta.indexes`, where each
entry is a tuple of fields and whether it is unique:


In [5]:
class Event(CatalogModel):
    tag = CharField(max_length=30)
    when = DateField()
    note = CharField(max_length=80, null=True)

    class Meta:
        indexes = ((("tag", "when"), True),)                        # the pair, and True for unique


db.create_tables([Event])
print(table_sql(Event))

Event.create(tag="restock", when=datetime.date(2026, 1, 5))
Event.create(tag="restock", when=datetime.date(2026, 1, 6))         # same tag, another day
try:
    Event.create(tag="restock", when=datetime.date(2026, 1, 5))
except peewee.IntegrityError as error:
    print(message(error))


CREATE TABLE IF NOT EXISTS "event" ("id" INTEGER NOT NULL PRIMARY KEY, "tag" VARCHAR(30) NOT NULL, "when" DATE NOT NULL, "note" VARCHAR(80))
CREATE UNIQUE INDEX IF NOT EXISTS "event_tag_when" ON "event" ("tag", "when")
peewee.IntegrityError: UNIQUE constraint failed: event.tag, event.when


The same tag twice is fine and the same tag on the same day is not, which is a rule neither column
could have carried alone. It is the same kind of thing as `unique=True`, a unique index, over two
columns instead of one, and the message names both of them.

`Meta.primary_key = CompositeKey("tag", "when")` says something stronger: that the pair **is** the
row's identity, and then there is no `id` column at all. It has a trap worth meeting before you need
it, and the **Creating and Changing Rows** notebook is where it is taken apart:


In [6]:
class Reading(CatalogModel):
    sensor = CharField(max_length=20)
    at = DateTimeField()
    value = IntegerField()

    class Meta:
        primary_key = CompositeKey("sensor", "at")


db.create_tables([Reading])
row = Reading(sensor="north", at=datetime.datetime(2026, 1, 5, 9, 0), value=11)
print("save() returned:", row.save(), "| rows in the table:", Reading.select().count())
print("with force_insert:", row.save(force_insert=True), "| rows:", Reading.select().count())


save() returned: 0 | rows in the table: 0
with force_insert: 1 | rows: 1


`save()` returned 0 and wrote nothing, with no error at all. peewee saw a primary key that already
had a value and read that as a change to an existing row, so it sent an `UPDATE` which matched
nothing. `force_insert=True` is the answer, and the reason this appears here rather than only in the
**Creating and Changing Rows** notebook is that it is a property of the key, not of one field type.

### What a value is when it comes back

Most fields give back what they took. Two are worth checking, because SQLite stores both as text:


In [7]:
class Payment(CatalogModel):
    amount = DecimalField(max_digits=8, decimal_places=3)
    at = DateTimeField()


db.create_tables([Payment])
Payment.create(amount=Decimal("1.005"), at=datetime.datetime(2026, 3, 1, 9, 30))

kept = Payment.get_by_id(1)
print("the decimal :", repr(kept.amount), "| unrounded:", kept.amount == Decimal("1.005"))
print("the datetime:", repr(kept.at))

db.execute_sql("INSERT INTO payment (amount, at) VALUES (?, ?)", ("2.50", "whenever"))
print("written by something else:", repr(Payment.get_by_id(2).at))


the decimal : Decimal('1.005') | unrounded: True
the datetime: datetime.datetime(2026, 3, 1, 9, 30)
written by something else: 'whenever'


The `Decimal` came back with all three of its digits, unrounded: `DecimalField` stores the number
rather than a float, which is why money belongs in it. The `datetime` came back a `datetime`, because
peewee converted the text it stored.

The last line is the one to remember. peewee converts what it wrote, and it cannot convert what it
did not: a row written by another program, a migration or a line of SQL comes back as whatever text
is in the column. The field is a promise about your own writes.

### The connection, opened and closed on purpose

A database object is closed until something needs it, and `autoconnect` is what opens it. The
catalog's own connection has been open since its first query, so this uses a database of its own:


In [8]:
lifecycle = SqliteDatabase(":memory:")                              # a database of its own, never used yet

print("autoconnect    :", lifecycle.autoconnect, "| open:", not lifecycle.is_closed())
print("connect()      :", lifecycle.connect(), "| open:", not lifecycle.is_closed())
try:
    lifecycle.connect()
except peewee.OperationalError as error:
    print("connect() again:", message(error))
print("reuse_if_open  :", lifecycle.connect(reuse_if_open=True))
print("close()        :", lifecycle.close(), "| open:", not lifecycle.is_closed())


autoconnect    : True | open: False
connect()      : True | open: True
connect() again: peewee.OperationalError: Connection already opened.
reuse_if_open  : False
close()        : True | open: False


`connect()` opens one and returns `True`, a second `connect()` raises, and
`connect(reuse_if_open=True)` returns `False` to say it did nothing, which is what a function that
may or may not be inside an open connection should use.

In a script, leaving `autoconnect` on is right. In anything long-running, a connection opened and
closed around a unit of work is better, and `connection_context` is that:


In [9]:
with lifecycle.connection_context():
    print("inside the block :", not lifecycle.is_closed())
print("after the block  :", not lifecycle.is_closed())


inside the block : True
after the block  : False


The block opened a connection, closed it afterwards, and would have closed it just as surely if
something inside had raised. A database made with `autoconnect=False` refuses to work without one,
which is the last of the Common errors and a good setting for a program where an unopened connection
should be a mistake rather than a surprise.

### The catalog's models, finished

The two models the rest of this guide uses, and the catalog loaded into them:


In [10]:
class Book(CatalogModel):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="books")
    year = IntegerField(index=True)
    pages = IntegerField()


db.create_tables([Book])
print(table_sql(Book))

Author.delete().execute()                                           # start from an empty catalog
with db.atomic():
    Author.insert_many([{"name": name, "first_book": year} for name, year in AUTHORS]).execute()
    written = {author.name: author.id for author in Author.select()}
    Book.insert_many([{"title": title, "author": written[author], "year": year, "pages": pages}
                      for title, author, year, pages in BOOKS]).execute()

print("loaded:", Author.select().count(), "authors and", Book.select().count(), "books")


CREATE TABLE IF NOT EXISTS "book" ("id" INTEGER NOT NULL PRIMARY KEY, "title" VARCHAR(80) NOT NULL, "author_id" INTEGER NOT NULL, "year" INTEGER NOT NULL, "pages" INTEGER NOT NULL, FOREIGN KEY ("author_id") REFERENCES "author" ("id"))
CREATE INDEX IF NOT EXISTS "book_author_id" ON "book" ("author_id")
CREATE INDEX IF NOT EXISTS "book_year" ON "book" ("year")
loaded: 4 authors and 12 books


`ForeignKeyField(Author, ...)` made a column called `author_id` and a constraint pointing at
`author.id`, and named the attribute `author`, which is the row rather than the number. It also made
an index on that column without being asked, which is what a join reads. The `backref="books"` put a
`books` attribute on `Author`. The **Relationships** notebook is where both
of those are taken apart, including the pragma SQLite needs before that constraint is enforced at
all.

### Where each part came from

| In the catalog's models | What it relies on | The section that showed it |
|---|---|---|
| `class CatalogModel(Model)` with `Meta.database` | one place that names the database | The base class |
| `CharField(max_length=80)`, `IntegerField(index=True)` | fields that become columns and indexes | What each field becomes |
| `unique=True` on the author's name | a rule the database keeps | `null`, `unique` and `index` |
| `ForeignKeyField(Author, backref="books")` | a column, a constraint and two attributes | The catalog's models |
| `db.create_tables([...])` | tables made once, in foreign key order | The base class |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/02-models-and-fields-solutions.ipynb).

**1.** Write a `Shelf` model on the catalog's base class, with a `code` that is unique, a `floor`
that is indexed, and a `note` that may be empty, and print its `CREATE TABLE` and index.


In [11]:
# your code here


**2.** Make the table, write two shelves, and show what the database does with a second shelf of the
same code.


In [12]:
# your code here


**3.** Write a `Copy` model whose table is called `stock` rather than `copy`, and print the name
peewee uses.


In [13]:
# your code here


**4.** Give `Copy` a rule that a shelf may hold one copy of a book, with `Meta.indexes`, and show it
refusing the second.


In [14]:
# your code here


**5.** Write a row into `payment` with plain SQL whose `amount` is text, read it back through the
model, and print what the field gives you.


In [15]:
# your code here


**6.** Open a connection with `connection_context`, count the books inside the block, and show the
connection is closed afterwards.


In [16]:
# your code here


## Common errors

### peewee.InterfaceError: Query must be bound to a database in order to call "execute".


In [17]:
class Loose(Model):                                                 # no Meta, so no database
    name = CharField()


print("the database it carries:", Loose._meta.database)
Loose.create(name="nowhere to go")


the database it carries: None


InterfaceError: Query must be bound to a database in order to call "execute".

A model with no `Meta.database` is bound to nothing, and a query built from it has nowhere to go.
The message says so precisely once it is read as a sentence about binding rather than about
execution.

Three ways in: forgetting `Meta` entirely, writing `Meta` but not inheriting the base class, or
misspelling `database`. All three leave `_meta.database` as `None`, which is the thing to print when
this happens:


In [18]:
class Found(CatalogModel):                                          # inherits the base, and its database
    name = CharField()


db.create_tables([Found])
print("bound to the database:", Found._meta.database is db, "| written:", Found.create(name="fine").id)


bound to the database: True | written: 1


### peewee.OperationalError: no such table: late


In [19]:
class Late(CatalogModel):
    name = CharField()


Late.create(name="the table was never made")


OperationalError: no such table: late

The twin of the first, and the one that arrives second: the model is bound, the database is open,
and nothing ever wrote the table. Declaring a class creates no table; `create_tables` does.

In a program the usual cause is a model that no import reaches before `create_tables` runs, since
peewee can only create the classes it has been handed:


In [20]:
db.create_tables([Late])
print("made now:", Late.create(name="written").id)


made now: 1


### peewee.OperationalError: table "late" already exists


In [21]:
db.create_tables([Late], safe=False)


OperationalError: table "late" already exists

`create_tables` writes `CREATE TABLE IF NOT EXISTS` by default, which is why running it at every
start is normal and harmless. `safe=False` drops the `IF NOT EXISTS`, and then a second run is an
error.

What it never does is change a table that is already there, whatever the models now say. That is the
**Migrations** notebook's subject, and it is worth knowing early, because a schema that changed and a
`create_tables` that "ran fine" is a confusing morning.

### peewee.IntegrityError: NOT NULL constraint failed: loan.borrower


In [22]:
Loan.create(copies=1, fee=Decimal("0.00"), taken_on=datetime.date(2026, 1, 5),
            due_at=datetime.datetime(2026, 2, 5, 9, 0))


IntegrityError: NOT NULL constraint failed: loan.borrower

`borrower` has no `null=True`, so the column is `NOT NULL`, and leaving it out means writing nothing
into a column that accepts nothing. The same message comes from a misspelled keyword, which is worth
knowing because that one looks nothing like a missing argument:


In [23]:
try:
    Loan.create(borrwer="Ines O'Brien", copies=1, fee=Decimal("0.00"),
                taken_on=datetime.date(2026, 1, 5), due_at=datetime.datetime(2026, 2, 5, 9, 0))
except peewee.IntegrityError as error:
    print(message(error))


peewee.IntegrityError: NOT NULL constraint failed: loan.borrower


peewee took `borrwer` as an ordinary attribute, set it on the object, and wrote a row with no
`borrower` in it. The column is what caught the typo, which is an argument for letting columns be
`NOT NULL` wherever a value is really required.

### peewee.InterfaceError: Error, database connection not opened.


In [24]:
strict = SqliteDatabase(":memory:", autoconnect=False)


class Strict(Model):
    name = CharField()

    class Meta:
        database = strict


strict.create_tables([Strict])


InterfaceError: Error, database connection not opened.

`autoconnect=False` says that a connection is opened on purpose or not at all. That is the right
setting for a long-running program, where a query on an unopened connection is a bug rather than
something to paper over, and the failure is this one rather than a connection quietly appearing.

The fix is to say when:


In [25]:
with strict.connection_context():
    strict.create_tables([Strict])
    print("inside :", Strict.create(name="on purpose").id, "| open:", not strict.is_closed())
print("after  : open:", not strict.is_closed())


inside : 1 | open: True
after  : open: False


## Recap

- Every peewee program has a base model whose `Meta` names the database, and every other model
  inherits it; a model with no database raises `InterfaceError` at the first query.
- The fields are the schema: peewee writes the `CREATE TABLE` from them, and `id` is added unless a
  primary key is declared.
- `null=True` and `unique=True` are rules the database keeps, `default=` is filled in by Python, and
  `index=True` is a statement of its own.
- `Meta.indexes` carries a rule about two columns together, and
  `Meta.primary_key = CompositeKey(...)` makes the pair the identity, with the `save()` trap that
  comes with it.
- `create_tables` is safe to run again and never changes a table that exists; a connection opens by
  itself unless `autoconnect=False`, and `connection_context` opens and closes one around a block.


## What is next

The **Creating and Changing Rows** notebook is about writing: `create` and `save`, the `save()` that
returns 0 and writes nothing, `insert_many` with the batch size that is not optional, `get_or_create`
and the defaults it ignores, upserts with `on_conflict`, and the delete that peewee 4 refuses to run
from an instance.


---

&#8592; **Previous:** [Why Peewee](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/01-why-peewee.ipynb)  &nbsp;·&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
